In [2]:
import os
import glob
import csv
import random
import time

import numpy as np
import h5py

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm


# ============================================================
# 1. Configuration
# ============================================================

# ------------------------------------------------------------
# Original training dataset
# ------------------------------------------------------------

DATA_ROOT = r"Training dataset"


# ------------------------------------------------------------
# IMPORTANT:
#
# Change this to the folder containing:
#
#     best_fno2d.pt
#
# Example:
#
# RUN_DIR =
# r"waveloss_mat_dataset_results/run_20260827_103852"
# ------------------------------------------------------------

RUN_DIR = "waveloss_mat_dataset_results/run_20260822_202638"


# ------------------------------------------------------------
# Trained model
# ------------------------------------------------------------

MODEL_PATH = os.path.join(
    RUN_DIR,
    "best_fno2d.pt"
)


# ------------------------------------------------------------
# Output files
# ------------------------------------------------------------

SUMMARY_PATH = os.path.join(
    RUN_DIR,
    "quantitative_metrics.txt"
)

CSV_PATH = os.path.join(
    RUN_DIR,
    "per_sample_metrics.csv"
)


# ============================================================
# 2. Dataset split
#
# MUST be identical to training
# ============================================================

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

SEED = 42


# ============================================================
# 3. FNO architecture
#
# MUST be identical to training
# ============================================================

WIDTH = 64

MODES_T = 96
MODES_X = 48

N_LAYERS = 4

IN_CHANNELS = 1
OUT_CHANNELS = 1


# ============================================================
# 4. Data dimensions
# ============================================================

NT = 501
NX = 200


# ============================================================
# Evaluation batch size
# ============================================================

BATCH_SIZE = 8


# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 72)
print("FNO QUANTITATIVE EVALUATION")
print("=" * 72)

print("\nUsing device:")
print(DEVICE)


# ============================================================
# 5. Check model file
# ============================================================

if not os.path.exists(
    MODEL_PATH
):

    raise FileNotFoundError(

        "\nCannot find trained FNO model:\n"
        f"{MODEL_PATH}\n\n"
        "Please modify RUN_DIR."
    )


print("\nTrained model:")
print(MODEL_PATH)


# ============================================================
# 6. Random seed
# ============================================================

random.seed(
    SEED
)

np.random.seed(
    SEED
)

torch.manual_seed(
    SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


# ============================================================
# 7. Find all .mat files
# ============================================================

all_files = glob.glob(

    os.path.join(
        DATA_ROOT,
        "**",
        "*.mat"
    ),

    recursive=True
)


# ============================================================
# IMPORTANT
#
# Original FNO training script:
#
# all_files = sorted(all_files)
# random.seed(42)
# random.shuffle(all_files)
#
# We reproduce exactly the same procedure.
# ============================================================

all_files = sorted(
    all_files
)


print(
    "\nTotal .mat samples found:",
    len(all_files)
)


if len(all_files) == 0:

    raise RuntimeError(

        f"No MAT files found under:\n"
        f"{DATA_ROOT}"
    )


# ============================================================
# 8. Reproduce original train / val / test split
# ============================================================

random.shuffle(
    all_files
)


n_total = len(
    all_files
)


n_train = int(
    n_total
    *
    TRAIN_RATIO
)


n_val = int(
    n_total
    *
    VAL_RATIO
)


train_files = all_files[
    :n_train
]


val_files = all_files[
    n_train:
    n_train + n_val
]


test_files = all_files[
    n_train + n_val:
]


print("\nDataset split:")

print(
    "Train samples:",
    len(train_files)
)

print(
    "Val samples  :",
    len(val_files)
)

print(
    "Test samples :",
    len(test_files)
)


# ============================================================
# 9. Compute normalization statistics
#
# IMPORTANT:
# Training data ONLY
#
# This reproduces the original FNO training normalization.
# ============================================================

def compute_statistics(
    file_list
):

    pt_sum = 0.0
    pt_sq_sum = 0.0

    pa_sum = 0.0
    pa_sq_sum = 0.0

    n_elements = 0


    print(
        "\nCalculating normalization statistics "
        "from training set..."
    )


    for mat_path in tqdm(

        file_list,

        desc="Statistics",

        ncols=100

    ):


        with h5py.File(
            mat_path,
            "r"
        ) as f:

            PT = np.asarray(
                f["PT"],
                dtype=np.float64
            )

            PA = np.asarray(
                f["PA"],
                dtype=np.float64
            )


        # ====================================================
        # Ensure:
        #
        # [T, X] = [501, 200]
        # ====================================================

        if PT.shape == (
            200,
            501
        ):

            PT = PT.T


        if PA.shape == (
            200,
            501
        ):

            PA = PA.T


        if PT.shape != (
            NT,
            NX
        ):

            raise ValueError(

                f"Wrong PT shape in:\n"
                f"{mat_path}\n"
                f"Shape = {PT.shape}"
            )


        if PA.shape != (
            NT,
            NX
        ):

            raise ValueError(

                f"Wrong PA shape in:\n"
                f"{mat_path}\n"
                f"Shape = {PA.shape}"
            )


        pt_sum += PT.sum()

        pt_sq_sum += np.square(
            PT
        ).sum()


        pa_sum += PA.sum()

        pa_sq_sum += np.square(
            PA
        ).sum()


        n_elements += PT.size


    # ========================================================
    # Mean
    # ========================================================

    pt_mean = (
        pt_sum
        /
        n_elements
    )


    pa_mean = (
        pa_sum
        /
        n_elements
    )


    # ========================================================
    # Variance
    # ========================================================

    pt_var = (
        pt_sq_sum
        /
        n_elements
        -
        pt_mean ** 2
    )


    pa_var = (
        pa_sq_sum
        /
        n_elements
        -
        pa_mean ** 2
    )


    pt_var = max(
        pt_var,
        0.0
    )


    pa_var = max(
        pa_var,
        0.0
    )


    # ========================================================
    # Standard deviation
    # ========================================================

    pt_std = (
        np.sqrt(
            pt_var
        )
        +
        1e-8
    )


    pa_std = (
        np.sqrt(
            pa_var
        )
        +
        1e-8
    )


    return (
        pt_mean,
        pt_std,
        pa_mean,
        pa_std
    )


# ============================================================
# Calculate normalization
# ============================================================

(
    pt_mean,
    pt_std,
    pa_mean,
    pa_std

) = compute_statistics(
    train_files
)


print("\nNormalization statistics:")

print(
    f"PT mean = {pt_mean:.8e}"
)

print(
    f"PT std  = {pt_std:.8e}"
)

print(
    f"PA mean = {pa_mean:.8e}"
)

print(
    f"PA std  = {pa_std:.8e}"
)


# ============================================================
# 10. Dataset
# ============================================================

class MatHeatAcousticDataset(
    Dataset
):

    def __init__(
        self,
        file_list,
        pt_mean,
        pt_std,
        pa_mean,
        pa_std
    ):

        self.file_list = file_list

        self.pt_mean = pt_mean
        self.pt_std = pt_std

        self.pa_mean = pa_mean
        self.pa_std = pa_std


    def __len__(
        self
    ):

        return len(
            self.file_list
        )


    def __getitem__(
        self,
        idx
    ):

        mat_path = self.file_list[
            idx
        ]


        with h5py.File(
            mat_path,
            "r"
        ) as f:

            PT = np.asarray(
                f["PT"],
                dtype=np.float32
            )

            PA = np.asarray(
                f["PA"],
                dtype=np.float32
            )


        # ====================================================
        # Convert to [501, 200]
        # ====================================================

        if PT.shape == (
            200,
            501
        ):

            PT = PT.T


        if PA.shape == (
            200,
            501
        ):

            PA = PA.T


        if PT.shape != (
            NT,
            NX
        ):

            raise ValueError(
                f"Wrong PT shape: "
                f"{PT.shape}"
            )


        if PA.shape != (
            NT,
            NX
        ):

            raise ValueError(
                f"Wrong PA shape: "
                f"{PA.shape}"
            )


        # ====================================================
        # Normalize
        # ====================================================

        PT = (
            PT
            -
            self.pt_mean
        ) / self.pt_std


        PA = (
            PA
            -
            self.pa_mean
        ) / self.pa_std


        # ====================================================
        # [T,X]
        #
        # ->
        #
        # [C,T,X]
        # ====================================================

        PT = torch.tensor(
            PT,
            dtype=torch.float32
        ).unsqueeze(
            0
        )


        PA = torch.tensor(
            PA,
            dtype=torch.float32
        ).unsqueeze(
            0
        )


        return (
            PT,
            PA,
            mat_path
        )


# ============================================================
# 11. Test dataset
# ============================================================

test_dataset = MatHeatAcousticDataset(

    test_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


print(
    "\nTest loader ready."
)


# ============================================================
# 12. 2D Spectral Convolution
# ============================================================

class SpectralConv2d(
    nn.Module
):

    def __init__(
        self,
        in_channels,
        out_channels,
        modes_t,
        modes_x
    ):

        super().__init__()


        self.in_channels = (
            in_channels
        )

        self.out_channels = (
            out_channels
        )

        self.modes_t = (
            modes_t
        )

        self.modes_x = (
            modes_x
        )


        scale = (
            1
            /
            (
                in_channels
                *
                out_channels
            )
        )


        self.weights = nn.Parameter(

            scale
            *
            torch.randn(

                in_channels,

                out_channels,

                modes_t,

                modes_x,

                dtype=torch.cfloat
            )
        )


    def compl_mul2d(
        self,
        x,
        weights
    ):

        return torch.einsum(
            "bixy,ioxy->boxy",
            x,
            weights
        )


    def forward(
        self,
        x
    ):

        B, C, T, X = (
            x.shape
        )


        # ====================================================
        # FFT
        # ====================================================

        x_ft = torch.fft.rfftn(

            x,

            dim=(-2, -1)
        )


        # ====================================================
        # Output Fourier tensor
        # ====================================================

        out_ft = torch.zeros(

            B,

            self.out_channels,

            T,

            X // 2 + 1,

            device=x.device,

            dtype=torch.cfloat
        )


        mt = min(
            self.modes_t,
            T
        )


        mx = min(
            self.modes_x,
            X // 2 + 1
        )


        out_ft[
            :,
            :,
            :mt,
            :mx
        ] = self.compl_mul2d(

            x_ft[
                :,
                :,
                :mt,
                :mx
            ],

            self.weights[
                :,
                :,
                :mt,
                :mx
            ]
        )


        # ====================================================
        # Inverse FFT
        # ====================================================

        x = torch.fft.irfftn(

            out_ft,

            s=(
                T,
                X
            ),

            dim=(-2, -1)
        )


        return x


# ============================================================
# 13. FNO2D
# ============================================================

class FNO2D(
    nn.Module
):

    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        width=64,
        modes_t=96,
        modes_x=48,
        n_layers=4
    ):

        super().__init__()


        self.width = width


        # ====================================================
        # Input:
        #
        # PT + t coordinate + x coordinate
        # ====================================================

        self.fc0 = nn.Conv2d(

            in_channels + 2,

            width,

            kernel_size=1
        )


        self.spectral_layers = (
            nn.ModuleList()
        )


        self.pointwise_layers = (
            nn.ModuleList()
        )


        for _ in range(
            n_layers
        ):


            self.spectral_layers.append(

                SpectralConv2d(

                    width,

                    width,

                    modes_t,

                    modes_x
                )
            )


            self.pointwise_layers.append(

                nn.Conv2d(

                    width,

                    width,

                    kernel_size=1
                )
            )


        self.fc1 = nn.Conv2d(

            width,

            128,

            kernel_size=1
        )


        self.fc2 = nn.Conv2d(

            128,

            out_channels,

            kernel_size=1
        )


    # ========================================================
    # Coordinate grid
    # ========================================================

    def get_grid(
        self,
        shape,
        device
    ):

        B, C, T, X = shape


        t = torch.linspace(

            0,

            1,

            T,

            device=device
        )


        x = torch.linspace(

            0,

            1,

            X,

            device=device
        )


        tt, xx = torch.meshgrid(

            t,

            x,

            indexing="ij"
        )


        grid = torch.stack(

            [
                tt,
                xx
            ],

            dim=0
        )


        grid = (

            grid
            .unsqueeze(0)
            .repeat(
                B,
                1,
                1,
                1
            )
        )


        return grid


    # ========================================================
    # Forward
    # ========================================================

    def forward(
        self,
        x
    ):


        grid = self.get_grid(

            x.shape,

            x.device
        )


        x = torch.cat(

            [
                x,
                grid
            ],

            dim=1
        )


        x = self.fc0(
            x
        )


        for spec, pw in zip(

            self.spectral_layers,

            self.pointwise_layers

        ):


            x1 = spec(
                x
            )


            x2 = pw(
                x
            )


            x = F.gelu(
                x1
                +
                x2
            )


        x = F.gelu(

            self.fc1(
                x
            )
        )


        x = self.fc2(
            x
        )


        return x


# ============================================================
# 14. Create FNO model
# ============================================================

fno_model = FNO2D(

    in_channels=IN_CHANNELS,

    out_channels=OUT_CHANNELS,

    width=WIDTH,

    modes_t=MODES_T,

    modes_x=MODES_X,

    n_layers=N_LAYERS

).to(
    DEVICE
)


# ============================================================
# 15. Load trained FNO weights
# ============================================================

print(
    "\nLoading trained FNO..."
)


state_dict = torch.load(

    MODEL_PATH,

    map_location=DEVICE,

    weights_only=True
)


fno_model.load_state_dict(
    state_dict
)


fno_model.eval()


print(
    "FNO loaded successfully."
)


# ============================================================
# 16. Evaluation storage
# ============================================================

# ------------------------------------------------------------
# GLOBAL accumulators
# ------------------------------------------------------------

global_sse = 0.0

global_sae = 0.0

global_target_sq = 0.0

global_target_sum = 0.0

global_n = 0


# ------------------------------------------------------------
# Individual sample results
# ------------------------------------------------------------

per_sample_results = []


# ============================================================
# 17. Evaluation
# ============================================================

print(
    "\nEvaluating test set..."
)


start_time = time.time()


sample_counter = 0


with torch.no_grad():


    for (

        PT_norm,

        PA_norm,

        file_paths

    ) in tqdm(

        test_loader,

        desc="Testing",

        ncols=100

    ):


        # ====================================================
        # GPU input
        # ====================================================

        PT_norm = PT_norm.to(

            DEVICE,

            non_blocking=True
        )


        # ====================================================
        # Prediction
        # ====================================================

        pred_norm = fno_model(
            PT_norm
        )


        # ====================================================
        # CPU
        # ====================================================

        pred_norm = (

            pred_norm
            .detach()
            .cpu()
            .numpy()

        )


        PA_norm_np = (

            PA_norm
            .numpy()

        )


        # ====================================================
        # DENORMALIZATION
        #
        # IMPORTANT:
        #
        # Quantitative metrics are calculated
        # on ORIGINAL PA values.
        # ====================================================

        pred_real = (

            pred_norm

            *

            pa_std

            +

            pa_mean

        )


        PA_real = (

            PA_norm_np

            *

            pa_std

            +

            pa_mean

        )


        current_batch_size = (
            pred_real.shape[0]
        )


        # ====================================================
        # Per sample
        # ====================================================

        for j in range(
            current_batch_size
        ):


            sample_counter += 1


            # ------------------------------------------------
            # Flatten
            # ------------------------------------------------

            pred_j = (

                pred_real[j]
                .reshape(-1)
                .astype(
                    np.float64
                )

            )


            true_j = (

                PA_real[j]
                .reshape(-1)
                .astype(
                    np.float64
                )

            )


            error_j = (
                pred_j
                -
                true_j
            )


            # =================================================
            # MSE
            # =================================================

            mse_j = np.mean(
                error_j ** 2
            )


            # =================================================
            # MAE
            # =================================================

            mae_j = np.mean(

                np.abs(
                    error_j
                )

            )


            # =================================================
            # RMSE
            # =================================================

            rmse_j = np.sqrt(
                mse_j
            )


            # =================================================
            # Relative L2
            # =================================================

            rel_l2_j = (

                np.linalg.norm(
                    error_j
                )

                /

                (
                    np.linalg.norm(
                        true_j
                    )

                    +

                    1e-12
                )

            )


            # =================================================
            # R²
            # =================================================

            ss_res_j = np.sum(
                error_j ** 2
            )


            target_mean_j = np.mean(
                true_j
            )


            ss_tot_j = np.sum(

                (
                    true_j
                    -
                    target_mean_j
                )

                ** 2

            )


            r2_j = (

                1.0

                -

                ss_res_j

                /

                (
                    ss_tot_j
                    +
                    1e-12
                )

            )


            # =================================================
            # File name
            # =================================================

            file_name = (
                file_paths[j]
            )


            # =================================================
            # Save per sample
            # =================================================

            per_sample_results.append(

                [
                    sample_counter,
                    file_name,
                    mse_j,
                    mae_j,
                    rmse_j,
                    rel_l2_j,
                    r2_j
                ]

            )


            # =================================================
            # GLOBAL statistics
            # =================================================

            global_sse += np.sum(
                error_j ** 2
            )


            global_sae += np.sum(

                np.abs(
                    error_j
                )

            )


            global_target_sq += np.sum(
                true_j ** 2
            )


            global_target_sum += np.sum(
                true_j
            )


            global_n += (
                true_j.size
            )


# ============================================================
# 18. Global metrics
# ============================================================

global_mse = (

    global_sse

    /

    global_n

)


global_mae = (

    global_sae

    /

    global_n

)


global_rmse = np.sqrt(
    global_mse
)


# ============================================================
# Global relative L2
# ============================================================

global_rel_l2 = (

    np.sqrt(
        global_sse
    )

    /

    (
        np.sqrt(
            global_target_sq
        )

        +

        1e-12
    )

)


# ============================================================
# Global R²
#
# SST =
# sum(y²) - sum(y)² / N
# ============================================================

global_sst = (

    global_target_sq

    -

    (
        global_target_sum ** 2

        /

        global_n
    )

)


global_r2 = (

    1.0

    -

    global_sse

    /

    (
        global_sst

        +

        1e-12
    )

)


# ============================================================
# 19. Per-sample mean ± std
# ============================================================

metric_array = np.asarray(

    [
        row[2:]
        for row in per_sample_results
    ],

    dtype=np.float64
)


sample_mean = np.mean(

    metric_array,

    axis=0
)


sample_std = np.std(

    metric_array,

    axis=0
)


# ============================================================
# 20. Evaluation time
# ============================================================

elapsed_time = (

    time.time()

    -

    start_time

)


time_per_sample = (

    elapsed_time

    /

    len(
        test_dataset
    )

)


# ============================================================
# 21. Print final results
# ============================================================

print("\n")

print("=" * 72)

print(
    "FNO FINAL QUANTITATIVE RESULTS"
)

print(
    "Metrics calculated on DENORMALIZED PA"
)

print("=" * 72)


print(
    f"MSE      : "
    f"{global_mse:.8e}"
)


print(
    f"MAE      : "
    f"{global_mae:.8e}"
)


print(
    f"RMSE     : "
    f"{global_rmse:.8e}"
)


print(
    f"Rel. L2  : "
    f"{global_rel_l2:.8e}"
)


print(
    f"R²       : "
    f"{global_r2:.8f}"
)


print("-" * 72)


print(
    "Per-sample mean ± std"
)


print(
    f"MSE      : "
    f"{sample_mean[0]:.8e} "
    f"± "
    f"{sample_std[0]:.8e}"
)


print(
    f"MAE      : "
    f"{sample_mean[1]:.8e} "
    f"± "
    f"{sample_std[1]:.8e}"
)


print(
    f"RMSE     : "
    f"{sample_mean[2]:.8e} "
    f"± "
    f"{sample_std[2]:.8e}"
)


print(
    f"Rel. L2  : "
    f"{sample_mean[3]:.8e} "
    f"± "
    f"{sample_std[3]:.8e}"
)


print(
    f"R²       : "
    f"{sample_mean[4]:.8f} "
    f"± "
    f"{sample_std[4]:.8f}"
)


print("-" * 72)


print(
    f"Evaluation time     : "
    f"{elapsed_time:.2f} s"
)


print(
    f"Time per sample     : "
    f"{time_per_sample:.4f} s"
)


print("=" * 72)


# ============================================================
# 22. Save per-sample CSV
# ============================================================

with open(

    CSV_PATH,

    "w",

    newline=""

) as f:


    writer = csv.writer(
        f
    )


    writer.writerow(

        [
            "sample",
            "file",
            "mse",
            "mae",
            "rmse",
            "relative_l2",
            "r2"
        ]

    )


    writer.writerows(
        per_sample_results
    )


# ============================================================
# 23. Save summary TXT
# ============================================================

with open(

    SUMMARY_PATH,

    "w"

) as f:


    f.write(
        "FNO Quantitative Evaluation\n"
    )


    f.write(
        "Metrics calculated on "
        "DENORMALIZED PA\n\n"
    )


    f.write(
        "Global test-set metrics\n"
    )


    f.write(
        "=======================\n"
    )


    f.write(
        f"MSE      : "
        f"{global_mse:.8e}\n"
    )


    f.write(
        f"MAE      : "
        f"{global_mae:.8e}\n"
    )


    f.write(
        f"RMSE     : "
        f"{global_rmse:.8e}\n"
    )


    f.write(
        f"Rel. L2  : "
        f"{global_rel_l2:.8e}\n"
    )


    f.write(
        f"R2       : "
        f"{global_r2:.8f}\n"
    )


    f.write(
        "\nPer-sample mean +/- std\n"
    )


    f.write(
        "========================\n"
    )


    names = [

        "MSE",

        "MAE",

        "RMSE",

        "Rel. L2",

        "R2"

    ]


    for (
        name,
        mean,
        std

    ) in zip(

        names,

        sample_mean,

        sample_std

    ):


        f.write(

            f"{name:<8}: "

            f"{mean:.8e} "

            f"+/- "

            f"{std:.8e}\n"

        )


    f.write(
        "\nTiming\n"
    )


    f.write(
        "======\n"
    )


    f.write(

        f"Total evaluation time : "
        f"{elapsed_time:.4f} s\n"

    )


    f.write(

        f"Time per sample       : "
        f"{time_per_sample:.6f} s\n"

    )


# ============================================================
# 24. Finish
# ============================================================

print(
    "\nResults saved to:"
)


print(
    SUMMARY_PATH
)


print(
    CSV_PATH
)


print(
    "\nFNO quantitative evaluation complete."
)

FNO QUANTITATIVE EVALUATION

Using device:
cuda

Trained model:
waveloss_mat_dataset_results/run_20260822_202638/best_fno2d.pt

Total .mat samples found: 800

Dataset split:
Train samples: 640
Val samples  : 80
Test samples : 80

Calculating normalization statistics from training set...


Statistics:   0%|                                                           | 0/640 [00:00<?, ?it/s]


Normalization statistics:
PT mean = 3.13556727e+01
PT std  = 2.16473979e+01
PA mean = -2.10088660e+04
PA std  = 2.84782403e+05

Test loader ready.

Loading trained FNO...
FNO loaded successfully.

Evaluating test set...


Testing:   0%|                                                               | 0/10 [00:00<?, ?it/s]



FNO FINAL QUANTITATIVE RESULTS
Metrics calculated on DENORMALIZED PA
MSE      : 1.65676367e+10
MAE      : 3.23420135e+04
RMSE     : 1.28715332e+05
Rel. L2  : 3.90803067e-01
R²       : 0.84604155
------------------------------------------------------------------------
Per-sample mean ± std
MSE      : 1.65676367e+10 ± 4.63075612e+10
MAE      : 3.23420135e+04 ± 5.43129747e+04
RMSE     : 6.42677085e+04 ± 1.11522636e+05
Rel. L2  : 4.66692818e-01 ± 1.62163706e-01
R²       : 0.75025921 ± 0.18473773
------------------------------------------------------------------------
Evaluation time     : 12.12 s
Time per sample     : 0.1516 s

Results saved to:
waveloss_mat_dataset_results/run_20260822_202638/quantitative_metrics.txt
waveloss_mat_dataset_results/run_20260822_202638/per_sample_metrics.csv

FNO quantitative evaluation complete.
